In [17]:

# ============================================================
# STUDENT POPULATION DISTRIBUTION MAP
# 7,633 RECORDS
#
# LOCATION: Current residence (city / ZIP)
# SHAPE: Role
# COLOR: Permanent region
# SIZE: Number of records
#
# GEOCODING: ArcGIS
# VISUALIZATION: Folium
# ============================================================

import pandas as pd
import folium
import math
import html
import webbrowser
import os

from pathlib import Path
from tqdm.auto import tqdm

from geopy.geocoders import ArcGIS
from geopy.extra.rate_limiter import RateLimiter


DATA_PATH = Path(r"C:\Users\MARE\OneDrive\Documents\data311_project2\data\processed\population_addresses_cleaned_processed.csv"
)

OUTPUT_DIR = DATA_PATH.parent

CACHE_PATH = OUTPUT_DIR / "current_location_cache.csv"

OUTPUT_MAP = OUTPUT_DIR / "student_population_role_region_map.html"

SUMMARY_PATH = OUTPUT_DIR / "population_location_summary.csv"


df = pd.read_csv(
    DATA_PATH,
    dtype={
        "current_zip": "string"
    }
)

print(f"Loaded records: {len(df):,}")


# Validate columns

required_columns = [
    "pop_id",
    "role",
    "current_city",
    "current_state",
    "current_zip",
    "permanent_region"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")


#clean data

def clean_text(series):

    return (
        series.astype("string")
        .str.strip()
        .replace("", pd.NA)
    )


for col in required_columns[1:]:

    if col != "pop_id":
        df[col] = clean_text(df[col])


# Standardize role names

df["role"] = df["role"].str.strip().str.title()

# Standardize region names

df["permanent_region"] = (
    df["permanent_region"]
    .str.replace(r"\s+", "", regex=True)
    .str.strip()
)


# Standardize states

df["current_state"] = df["current_state"].str.upper()


# Standardize ZIP codes

df["current_zip"] = (
    df["current_zip"]
    .str.extract(r"(\d{1,5})", expand=False)
    .str.zfill(5)
)


# Known geographic correction

mck_mask = (
    df["current_city"]
    .str.contains("McKinleyville", case=False, na=False)
)

df.loc[mck_mask, "current_city"] = "McKinleyville"

df.loc[mck_mask, "current_state"] = "CA"


# standard roles

def standardize_role(role):

    role = str(role).strip().lower()

    if "faculty" in role:
        return "Faculty"

    if "student" in role:
        return "Student"

    if "staff" in role or "admin" in role:
        return "Staff/Admin"

    return "Other"


df["map_role"] = df["role"].apply(standardize_role)


# standard geo loc

region_mapping = {

    "NorCal": "NorCal",

    "SanDiego": "SanDiego",

    "LA": "LA",

    "OutOfState": "OutOfState",

    "BayArea": "BayArea"

}

df["map_region"] = df["permanent_region"].map(
    region_mapping
).fillna("Other")


# construct current loc

# Use city + state + ZIP when available.
# If city is missing, use ZIP + state.

df["location_query"] = (
    df["current_city"].fillna("")
    + ", "
    + df["current_state"].fillna("")
    + " "
    + df["current_zip"].fillna("")
).str.strip(" ,")


# Fallback for missing cities

missing_city = df["current_city"].isna()

df.loc[missing_city, "location_query"] = (
    df.loc[missing_city, "current_zip"].fillna("")
    + ", "
    + df.loc[missing_city, "current_state"].fillna("")
    + ", USA"
)


# Remove records without sufficient location information

df = df[
    df["current_zip"].str.fullmatch(r"\d{5}", na=False)
    & df["current_state"].notna()
].copy()


print(f"Records with valid ZIP formatting: {len(df):,}")


# agg pop counts

group_columns = [

    "current_city",

    "current_state",

    "current_zip",

    "location_query",

    "map_role",

    "map_region"

]


population = (

    df.groupby(
        group_columns,
        dropna=False
    )

    .agg(
        population_count=("pop_id", "nunique")
    )

    .reset_index()

)


print("\nAGGREGATION RESULTS")
print("-" * 50)

print(f"Original records: {len(df):,}")

print(f"Aggregated markers: {len(population):,}")

print(
    f"Records represented: "
    f"{population['population_count'].sum():,}"
)


# ID unique loc for geo coding

locations = (

    population[
        [
            "location_query",
            "current_city",
            "current_state",
            "current_zip"
        ]
    ]

    .drop_duplicates("location_query")

    .reset_index(drop=True)

)


print(
    f"\nUnique current locations: {len(locations):,}"
)


# initialize geo encoder arcgis

geolocator = ArcGIS(timeout=15)

geocode = RateLimiter(

    geolocator.geocode,

    min_delay_seconds=0.2,

    max_retries=2,

    error_wait_seconds=5,

    swallow_exceptions=True,

    return_value_on_exception=None

)


# load geocode caceh

if CACHE_PATH.exists():

    cache = pd.read_csv(CACHE_PATH)

else:

    cache = pd.DataFrame(
        columns=[
            "location_query",
            "latitude",
            "longitude"
        ]
    )


cache = cache.dropna(
    subset=["latitude", "longitude"]
)

cache = cache.drop_duplicates(
    subset=["location_query"],
    keep="last"
)


cached_locations = set(
    cache["location_query"]
)


new_locations = [

    location

    for location in locations["location_query"]

    if location not in cached_locations

]


# geocode unique loc..

new_results = []

print("\nGEOCODING CURRENT LOCATIONS")


for i, address in enumerate(

    tqdm(
        new_locations,
        desc="Geocoding"
    ),

    start=1

):

    try:

        result = geocode(address)

        if result:

            new_results.append({

                "location_query": address,

                "latitude": result.latitude,

                "longitude": result.longitude

            })


    except Exception as error:

        print(f"\nError: {address} | {error}")


    # Save every 25 locations

    if i % 25 == 0 and new_results:

        cache = pd.concat(

            [
                cache,
                pd.DataFrame(new_results)
            ],

            ignore_index=True

        )

        cache = cache.drop_duplicates(
            subset=["location_query"],
            keep="last"
        )

        cache.to_csv(
            CACHE_PATH,
            index=False
        )

        new_results = []


# Save remaining results

if new_results:

    cache = pd.concat(

        [
            cache,
            pd.DataFrame(new_results)
        ],

        ignore_index=True

    )


cache.to_csv(
    CACHE_PATH,
    index=False
)


# merge coordinate

population = population.merge(

    cache[
        [
            "location_query",
            "latitude",
            "longitude"
        ]
    ],

    on="location_query",

    how="left"

)


# Identify failed geocoding records

unmatched = population[
    population["latitude"].isna()
    |
    population["longitude"].isna()
].copy()


print(
    f"\nUnmatched aggregated markers: {len(unmatched):,}"
)


# Remove unmatched locations from map

population = population.dropna(
    subset=["latitude", "longitude"]
).copy()


# marker color

region_colors = {

    "NorCal": "green",

    "SanDiego": "blue",

    "LA": "red",

    "OutOfState": "purple",

    "BayArea": "orange",

    "Other": "gray"

}


# marker shape

role_icons = {

    "Student": "circle",

    "Staff/Admin": "square-full",

    "Faculty": "triangle"

}


# map detail

m = folium.Map(

    location=[40.8665, -124.0828],

    zoom_start=7,

    tiles="CartoDB positron",

    control_scale=True

)


# marker cluster

from folium.plugins import MarkerCluster

marker_cluster = MarkerCluster(

    name="Population Distribution",

    disableClusteringAtZoom=12

).add_to(m)


# add agg. markers

for row in tqdm(

    population.itertuples(index=False),

    total=len(population),

    desc="Building map"

):

    color = region_colors.get(
        row.map_region,
        "gray"
    )


    shape = role_icons.get(
        row.map_role,
        "circle"
    )


    # Scale markers based on population count

    radius = min(

        18,

        max(

            5,

            4 + 3 * math.log1p(
                row.population_count
            )

        )

    )


    # Popup without personally identifying information

    popup_html = f"""

    <div style="font-size:13px; width:200px">

        <b>Role:</b> {html.escape(row.map_role)}<br>

        <b>Permanent Region:</b>
        {html.escape(row.map_region)}<br>

        <b>Current Location:</b>
        {html.escape(str(row.current_city))}<br>

        <b>ZIP:</b>
        {html.escape(str(row.current_zip))}<br>

        <b>Population Count:</b>
        {row.population_count:,}

    </div>

    """


    folium.Marker(

        location=[
            row.latitude,
            row.longitude
        ],

        tooltip=(

            f"{row.map_role} | "

            f"{row.map_region} | "

            f"Count: {row.population_count:,}"

        ),

        popup=folium.Popup(
            popup_html,
            max_width=250
        ),

        icon=folium.DivIcon(

            html=f"""

            <div style="

                width:{radius * 2}px;

                height:{radius * 2}px;

                background-color:{color};

                border:2px solid white;

                border-radius:{
                    '50%' if shape == 'circle'
                    else '0' if shape == 'square-full'
                    else '0'
                };

                clip-path:{
                    'polygon(50% 0%, 0% 100%, 100% 100%)'
                    if shape == 'triangle'
                    else 'none'
                };

                opacity:0.85;

                box-shadow:0 1px 3px #555;

            "></div>

            """,

            icon_size=(
                radius * 2,
                radius * 2
            ),

            icon_anchor=(
                radius,
                radius
            )

        )

    ).add_to(marker_cluster)


# create legend

legend_html = """

<div style="

position: fixed;

bottom: 30px;

left: 30px;

width: 245px;

background-color: white;

border: 1px solid #ccc;

border-radius: 6px;

padding: 12px;

z-index: 9999;

font-size: 12px;

box-shadow: 0 2px 6px rgba(0,0,0,0.2);

">

<b>POPULATION DISTRIBUTION</b>

<hr>

<b>ROLE — SHAPE</b><br>

● Student<br>

■ Staff / Admin<br>

▲ Faculty<br>

<hr>

<b>PERMANENT REGION — COLOR</b><br>

<span style="color:green">●</span> NorCal<br>

<span style="color:blue">●</span> San Diego<br>

<span style="color:red">●</span> LA<br>

<span style="color:purple">●</span> OutOfState<br>

<span style="color:orange">●</span> BayArea<br>

<hr>

<b>MARKER SIZE</b><br>

Larger marker = More records

</div>

"""

m.get_root().html.add_child(
    folium.Element(legend_html)
)


# map 

m.save(OUTPUT_MAP)


# export summeyr

summary = (

    population

    .groupby(
        [
            "map_role",
            "map_region"
        ],

        as_index=False
    )

    .agg(
        population_count=(
            "population_count",
            "sum"
        )
    )

)


summary.to_csv(
    SUMMARY_PATH,
    index=False
)


# final output

print("\n")
print("=" * 60)

print("MAP SUCCESSFULLY CREATED")

print("=" * 60)

print(f"Original records: {len(df):,}")

print(
    f"Aggregated markers: {len(population):,}"
)

print(
    f"Records represented: "
    f"{population['population_count'].sum():,}"
)

print(
    f"Unmatched markers: {len(unmatched):,}"
)

print("\nOUTPUT FILES")

print(f"Map: {OUTPUT_MAP}")

print(f"Cache: {CACHE_PATH}")

print(f"Summary: {SUMMARY_PATH}")

print("=" * 60)


Loaded records: 7,633
Records with valid ZIP formatting: 7,633

AGGREGATION RESULTS
--------------------------------------------------
Original records: 7,633
Aggregated markers: 42
Records represented: 7,633

Unique current locations: 6

GEOCODING CURRENT LOCATIONS


Geocoding: 0it [00:00, ?it/s]


Unmatched aggregated markers: 0


c:\Users\MARE\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\folium\raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


Building map:   0%|          | 0/42 [00:00<?, ?it/s]



MAP SUCCESSFULLY CREATED
Original records: 7,633
Aggregated markers: 42
Records represented: 7,633
Unmatched markers: 0

OUTPUT FILES
Map: C:\Users\MARE\OneDrive\Documents\data311_project2\data\processed\student_population_role_region_map.html
Cache: C:\Users\MARE\OneDrive\Documents\data311_project2\data\processed\current_location_cache.csv
Summary: C:\Users\MARE\OneDrive\Documents\data311_project2\data\processed\population_location_summary.csv
